In [1]:
from pathlib import Path
import json
import pandas as pd

# Display settings (optional)
pd.set_option('display.max_colwidth', 120)

# How many rows to show when displaying DataFrames in the notebook
pd.set_option('display.max_rows', 200)  # increase if you want more
pd.set_option('display.min_rows', 50)

In [2]:
# Choose the input JSON file (JSON Whole Model export)
file_name = "ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json"

# Default: workspace-root/JSON Whole Model/<file_name> (works when notebook is in PyDataTransform/)
input_path = (Path('..') / 'JSON Whole Model' / file_name).resolve()
if not input_path.exists():
    input_path = (Path('JSON Whole Model') / file_name).resolve()

assert input_path.exists(), f"File not found: {input_path}"

# Copy the JSON being read into JSON_Edit for working edits
json_edit_dir = (Path('..') / 'JSON_Edit').resolve()
if not json_edit_dir.exists():
    json_edit_dir = Path('JSON_Edit').resolve()
json_edit_dir.mkdir(parents=True, exist_ok=True)

working_json_path = json_edit_dir / input_path.name
working_json_path.write_text(input_path.read_text(encoding='utf-8'), encoding='utf-8')
print(f"Copied source JSON to: {working_json_path}")

with working_json_path.open('r', encoding='utf-8') as f:
    records = json.load(f)

df = pd.DataFrame(records)
print(f"Working DataFrame shape: {df.shape}")

Copied source JSON to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
Working DataFrame shape: (6580, 4)


#### Model element Name counts
This notebook loads a `JSON Whole Model/*.json` export and prints a table with **Name**, **DbId**, **GUID**, and the **total count of that Name** across the model (duplicates included).

In [3]:
def _extract_guid(props):
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

required_columns = {'Name', 'DbId', 'Properties'}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

# Build GUID + name counts table
df['GUID'] = df['Properties'].apply(_extract_guid)
name_counts = df['Name'].value_counts(dropna=False)
df['NameCount'] = df['Name'].map(name_counts)

table = (
    df[['Name', 'DbId', 'GUID', 'NameCount']]
    .sort_values(['Name', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

rows_to_show = 50  # set to None to display all rows
print(f"Total rows in table: {len(table)}")
table if rows_to_show is None else table.head(rows_to_show)

Total rows in table: 6580


,Name,DbId,GUID,NameCount
0,#4662766,1013,None,1
1,#4662804,1689,None,1
2,1JNL100332-220_00-Rubber Plate,5014,225154f9-31fd-32e5-b35f-82d1d1a7165a,288
3,1JNL100332-220_00-Rubber Plate,5015,651ab0ea-4446-3764-b7b9-657736c28330,288
4,1JNL100332-220_00-Rubber Plate,5016,74c01236-6f37-32c9-a625-306be1debbb3,288
5,1JNL100332-220_00-Rubber Plate,5017,8d88dfc1-242d-3f6f-839c-dc416b312455,288
6,1JNL100332-220_00-Rubber Plate,5034,edc2be86-7af8-3602-8c45-b2a785e8bcc6,288
7,1JNL100332-220_00-Rubber Plate,5035,5335ee00-1166-39b0-a6f6-7a89354a41e1,288
8,1JNL100332-220_00-Rubber Plate,5036,74542c6e-6eb0-3454-8421-3adb1cb79c53,288
9,1JNL100332-220_00-Rubber Plate,5037,2dc03adc-5b58-31c5-8d4a-814d14f9c55b,288


In [6]:
import re

def _extract_guid(props):
    # `props` is the element's `Properties` array.
    # We look for the entry with displayName == 'GUID' (case-insensitive).
    if not isinstance(props, list):
        return None
    for item in props:
        if not isinstance(item, dict):
            continue
        display_name = str(item.get('displayName', '')).strip()
        if display_name.lower() == 'guid':
            return item.get('value')
    return None

def _strip_guid_prefix(name):
    if pd.isna(name):
        return name
    text = str(name)

    # Trim prefixes like "1JNL100332-220_00-Rubber Plate" -> "00-Rubber Plate".
    # Rule: if there is one underscore and the part before it looks like an ID-like token
    # (contains at least one digit and optional hyphens), keep only the part after underscore.
    match = re.match(r'^([A-Za-z0-9-]+)_(.+)$', text)
    if not match:
        return text

    prefix, rest = match.groups()
    if any(ch.isdigit() for ch in prefix):
        return rest
    return text

def _write_json_unescaped(path, payload):
    # Python json.dumps keeps slashes as '/' (does not force '\\/')
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

# Keep old names for comparison
df['OldName'] = df['Name']
df['NewName'] = df['Name'].apply(_strip_guid_prefix)
df['Name'] = df['NewName']

# Sync cleaned names back to original JSON records
for idx, item in enumerate(records):
    if idx < len(df):
        item['Name'] = None if pd.isna(df.at[idx, 'NewName']) else str(df.at[idx, 'NewName'])

# Save amended JSON into JSON_Edit (does not change original source file)
_write_json_unescaped(working_json_path, records)
print(f"Updated JSON written to: {working_json_path}")

# Build GUID + key columns (GUID is primary unique key for matching)
df['GUID'] = df['Properties'].apply(_extract_guid)
df['ElementKey'] = df['GUID'].fillna(df['ExternalId'])
name_counts = df['NewName'].value_counts(dropna=False)
df['NameCount'] = df['NewName'].map(name_counts)

guid_duplicates = df['GUID'].dropna().duplicated().sum()
print(f"GUID duplicates found: {guid_duplicates}")

table = (
    df[['ElementKey', 'GUID', 'ExternalId', 'OldName', 'NewName', 'DbId', 'NameCount']]
    .sort_values(['NewName', 'DbId'], kind='stable')
    .reset_index(drop=True)
)

# 4) Print out the updated table with old/new name comparison
rows_to_show = 50  # set to None to show all rows (can be slow/huge)
table if rows_to_show is None else table.head(rows_to_show)

Updated JSON written to: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-MPD-TXBP1-M-M-0001.json
GUID duplicates found: 0


,ElementKey,GUID,ExternalId,OldName,NewName,DbId,NameCount
0,0/0/0/0/71/0/3,None,0/0/0/0/71/0/3,#4662766,#4662766,1013,1
1,0/0/0/0/121/0/2,None,0/0/0/0/121/0/2,#4662804,#4662804,1689,1
2,fa70f11f-de84-3523-98a2-b18abf9e15e2,fa70f11f-de84-3523-98a2-b18abf9e15e2,0/0/0/0/71,00-AC filter Current Transformer,00-AC filter Current Transformer,81,2
3,59cc019b-b880-3dc9-bb7b-84e7dbf935d8,59cc019b-b880-3dc9-bb7b-84e7dbf935d8,0/0/0/0/121,00-AC filter Current Transformer,00-AC filter Current Transformer,131,2
4,f2857974-3215-36e2-99d5-960065acf8b4,f2857974-3215-36e2-99d5-960065acf8b4,0/0/0/0/157/1/0/0,00-Aluminium Plate,00-Aluminium Plate,5013,108
5,ede72261-202d-3ad3-afeb-386f60cc0994,ede72261-202d-3ad3-afeb-386f60cc0994,0/0/0/0/157/1/1/0,00-Aluminium Plate,00-Aluminium Plate,5033,108
6,8aeaa601-9c8e-3abe-b17f-44aef63a6015,8aeaa601-9c8e-3abe-b17f-44aef63a6015,0/0/0/0/157/1/2/0,00-Aluminium Plate,00-Aluminium Plate,5053,108
7,0fcd21cc-3a89-3237-8b04-6b4c64ca4561,0fcd21cc-3a89-3237-8b04-6b4c64ca4561,0/0/0/0/157/1/3/0,00-Aluminium Plate,00-Aluminium Plate,5073,108
8,588dc17f-3eb2-38eb-818b-1c136986e693,588dc17f-3eb2-38eb-818b-1c136986e693,0/0/0/0/157/1/4/0,00-Aluminium Plate,00-Aluminium Plate,5093,108
9,d05c828c-263b-310c-babb-b90adaab48f3,d05c828c-263b-310c-babb-b90adaab48f3,0/0/0/0/157/1/5/0,00-Aluminium Plate,00-Aluminium Plate,5113,108
